# Genesis Manthan — Local Dev Setup

Smoke-test all project modules on CPU (no GPU required).  
Run this notebook before uploading anything to Kaggle.

**Pre-requisites**: venv activated, `pip install -r requirements-dev.txt`

In [ ]:
import sys, subprocess, importlib

# 1. Verify Python version
print(f'Python: {sys.version}')
assert sys.version_info >= (3, 10), 'Python 3.10+ required'

# 2. Verify critical imports
for pkg in ['transformers', 'datasets', 'peft', 'torch']:
    try:
        mod = importlib.import_module(pkg)
        ver = getattr(mod, '__version__', 'unknown')
        print(f'  ✓ {pkg} {ver}')
    except ImportError:
        print(f'  ✗ {pkg} NOT INSTALLED — run: pip install {pkg}')

import torch
print(f'  CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# 3. Smoke-test reward functions (CPU, no model)
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from src.training.reward_functions import (
    tool_execution_reward, answer_correctness_reward, format_reward, combined_reward
)

# Tool execution reward
r = tool_execution_reward('<tool_call>{"name": "python_repl", "arguments": {"code": "print(42)"}}</tool_call>')
assert 0.0 <= r <= 1.0, f'Expected [0,1], got {r}'
print(f'tool_execution_reward: {r:.2f} ✓')

# Answer correctness
r = answer_correctness_reward('<final_answer>42</final_answer>', '42')
assert r == 1.0, f'Expected 1.0, got {r}'
print(f'answer_correctness_reward (exact): {r:.2f} ✓')

# Format reward
r = format_reward('<tool_call>{"name":"x"}</tool_call>')
assert r == 0.1
print(f'format_reward: {r:.2f} ✓')

# Combined reward
completion = '<tool_call>{"name": "python_repl", "arguments": {"code": "print(42)"}}</tool_call>\n<final_answer>42</final_answer>'
r = combined_reward(completion, '42')
assert 0.0 <= r <= 1.0
print(f'combined_reward: {r:.2f} ✓')

print('\n✅ All reward function smoke tests passed')

In [ ]:
# 4. Smoke-test code sandbox
from src.training.grpo_train import execute_code_sandbox

tests = [
    ('print(2 + 2)', True, '4'),
    ('print("hello manthan")', True, 'hello manthan'),
    ('x = [i**2 for i in range(5)]\nprint(sum(x))', True, '30'),
    ('raise ValueError("test error")', False, None),
    ('syntax error here !!', False, None),
]

all_passed = True
for code, expect_success, expect_output in tests:
    result = execute_code_sandbox(code)
    ok = result['success'] == expect_success
    if expect_output:
        ok = ok and result.get('result', '').strip() == expect_output
    status = '✓' if ok else '✗'
    print(f'  {status} {code[:50]!r}')
    if not ok:
        print(f'    Expected: success={expect_success}, output={expect_output}')
        print(f'    Got: {result}')
        all_passed = False

print(f'\n{"✅ All sandbox tests passed" if all_passed else "❌ Some sandbox tests failed"}')

In [ ]:
# 5. Smoke-test budget forcing (tokenizer only, no model)
from transformers import AutoTokenizer
from src.inference.budget_forcing import BudgetForcingProcessor

tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-1.5B-Instruct')

processor = BudgetForcingProcessor(
    tokenizer=tokenizer,
    min_tool_calls=1,
    max_tool_calls=5,
)
print('BudgetForcingProcessor created ✓')
print(f'  final_answer_token_id: {processor.final_answer_token_id}')
print(f'  wait_token_id: {processor.wait_token_id}')
print('\n✅ Budget forcing smoke test passed')

In [ ]:
# 6. Smoke-test SFT training script
import subprocess, sys
result = subprocess.run(
    [sys.executable, '../src/training/sft_train.py', '--smoke-test'],
    capture_output=True, text=True, timeout=60
)
print('STDOUT:', result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    print('❌ SFT smoke test FAILED')
else:
    print('✅ SFT smoke test passed')

In [ ]:
# 7. Smoke-test GRPO training script
result = subprocess.run(
    [sys.executable, '../src/training/grpo_train.py', '--smoke-test'],
    capture_output=True, text=True, timeout=60
)
print('STDOUT:', result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    print('❌ GRPO smoke test FAILED')
else:
    print('✅ GRPO smoke test passed')

In [ ]:
# 8. Run all pytest unit tests
result = subprocess.run(
    [sys.executable, '-m', 'pytest', '../tests/', '-v', '--tb=short'],
    capture_output=True, text=True, cwd='..'
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    print('❌ Some tests FAILED')
else:
    print('✅ All tests passed')

## Summary

If all cells above completed with ✅, the project is ready for Kaggle training.  

**Next steps:**
1. Generate synthetic data: `python src/data/generate_synthetic.py --num-problems 100 --output data/raw/synthetic.jsonl`
2. Format dataset: `python src/data/format_dataset.py --input data/raw/ --output data/processed/`
3. Upload `notebooks/02_sft_kaggle.ipynb` to Kaggle and run Phase 1 SFT
4. Upload `notebooks/03_grpo_kaggle.ipynb` to Kaggle and run Phase 2 GRPO